In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.datasets import load_iris, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, BisectingKMeans, AgglomerativeClustering, DBSCAN, OPTICS
from sklearn.metrics import pairwise_distances
from sklearn.metrics import rand_score, adjusted_rand_score
from sklearn.metrics import mutual_info_score, adjusted_mutual_info_score, normalized_mutual_info_score
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

In [ ]:
external = [rand_score, adjusted_rand_score, mutual_info_score, adjusted_mutual_info_score, normalized_mutual_info_score]
internal = [silhouette_score, calinski_harabasz_score, davies_bouldin_score]

def evaluate(category, name, labels):
    keep = labels >= 0
    X_keep, kept = X[keep], labels[keep]
    accuracy = pd.crosstab(kept, y[keep]).max(axis=1).sum() / len(y)
    sse = ssb = 0
    for c in set(kept):
        cluster = X_keep[kept == c]
        center = cluster.mean(axis=0)
        sse += ((cluster - center) ** 2).sum()
        ssb += len(cluster) * ((center - X_keep.mean(axis=0)) ** 2).sum()
    scores = [m(y, labels) for m in external] + [m(X_keep, kept) for m in internal]
    results.append([category, name, accuracy] + scores + [sse, ssb])

In [ ]:
def show():
    names = [m.__name__ for m in external + internal]
    table = pd.DataFrame(results, columns=["category", "algorithm", "accuracy"] + names + ["SSE", "SSB"])
    for category in ["partition", "hierarchical", "density"]:
        display(table[table["category"] == category].round(3))

In [ ]:
def k_medoids():
    distances = pairwise_distances(X)
    medoids = [0, 1, 2]
    best = distances[:, medoids].min(axis=1).sum()
    improved = True
    while improved:
        improved = False
        for i in range(3):
            for point in range(len(X)):
                trial = medoids.copy()
                trial[i] = point
                cost = distances[:, trial].min(axis=1).sum()
                if cost < best:
                    best, medoids, improved = cost, trial, True
    return distances[:, medoids].argmin(axis=1)

In [ ]:
def partition():
    evaluate("partition", "K-means", KMeans(3, init="random", n_init=10, random_state=42).fit_predict(X))
    evaluate("partition", "K-means++", KMeans(3, init="k-means++", n_init=10, random_state=42).fit_predict(X))
    evaluate("partition", "Bisecting K-means", BisectingKMeans(3, random_state=42).fit_predict(X))
    evaluate("partition", "K-medoids", k_medoids())

In [ ]:
def hierarchical():
    for method in ["ward", "complete", "average", "single"]:
        evaluate("hierarchical", method, AgglomerativeClustering(3, linkage=method).fit_predict(X))
        plt.figure(figsize=(10, 4))
        dendrogram(linkage(X, method), no_labels=True)
        plt.title(f"{method} dendrogram")
        plt.show()

In [ ]:
def density(eps, dbscan_samples, optics_samples):
    evaluate("density", "DBSCAN", DBSCAN(eps=eps, min_samples=dbscan_samples).fit_predict(X))
    evaluate("density", "OPTICS", OPTICS(min_samples=optics_samples, min_cluster_size=0.2, xi=0.01).fit_predict(X))

In [ ]:
dataset = load_iris()
X, y = dataset.data, dataset.target
print(dataset.target_names)
results = []

partition()
hierarchical()
density(0.4, 3, 5)
show()

In [ ]:
dataset = load_wine()
X, y = dataset.data, dataset.target
print(dataset.target_names)
X = PCA(2).fit_transform(StandardScaler().fit_transform(X))
results = []

partition()
hierarchical()
density(1, 20, 20)
show()

In [ ]:
for eps in [0.5, 0.7, 0.9, 1, 1.1, 1.3]:
    labels = DBSCAN(eps=eps, min_samples=20).fit_predict(X)
    print("eps", eps, "clusters", labels.max() + 1, "noise", (labels < 0).sum(), "ARI", adjusted_rand_score(y, labels))